<a href="https://colab.research.google.com/github/jac0bmath3w/rail-safety-ai/blob/main/notebooks/02_rag_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/jac0bmath3w/rail-safety-ai.git

In [ ]:
!pip install pypdf
!pip install langchain
!pip install langchain-community
!pip install langchain-huggingface
!pip install llama-index
!pip install -U langchain-text-splitters
!pip install chromadb
!pip install -U bitsandbytes>=0.46.1
!pip install --upgrade transformers
!pip install sentencepiece tiktoken

In [ ]:
import sys, os
src_path = os.path.join(os.getcwd(), 'rail-safety-ai', 'src')
sys.path.append(src_path)

In [ ]:
import torch
DB_PATH = "/content/drive/MyDrive/rail_ai_project/vector_db"


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from vector_store import RailVectorVault
from engine import RailSafetyEngine
from embed import RailEmbedder


In [ ]:
import importlib
import engine
importlib.reload(engine)
from engine import RailSafetyEngine

In [ ]:
embedder = RailEmbedder(model_name='BAAI/bge-base-en-v1.5')
vault = RailVectorVault(embedder_instance=embedder, db_path=DB_PATH)

print(f"Vault synchronized. Total Knowledge Base Chunks: {vault.collection.count()}")


In [ ]:
# engine = RailSafetyEngine(model_id = "google/gemma-4-31b-it") # Won't work on free colab
engine = RailSafetyEngine()
# engine = RailSafetyEngine(model_id = 'unsloth/gemma-4-31B-it-GGUF')

In [ ]:
# raw_question = "Are Highway-Rail Grade Crossing Collision fatalities or Trespassing Fatalities more common?"
# raw_question = "When can a crossing be considered for grade separation?"
# raw_question = "Who is the author of the HRGC handbook 2019 edition? What does HRGC stand for?"
# raw_question = 'what does HRGC stand for?'
# raw_question = 'what does adjacent track mean?'
raw_question = 'what enforcement tools are available when enforcing the hazardous materials regulations?'
# raw_question = 'A rural highway-rail grade crossing has these conditions: posted highway speed: 60 mph AADT: 18,500, freight trains per day: 28, maximum authorized train speed: 70 mph, expected accident frequency with gates: 0.3 per year, vehicle delay: 35 vehicle-hours per day, acceptable alternate access exists within 0.8 mile, closing the crossing would increase the median trip by 2.2 miles, Based on the Handbook’s Chapter 3 guidance, which of the following is best supported? A. The crossing should primarily be considered for closure only B. The crossing should primarily be considered for grade separation C. Neither closure nor grade separation is supported D. Both closure and grade separation are supported by the listed criteria'

In [ ]:
search_results = vault.query(raw_question, n_results=15)

In [ ]:
chunks = search_results['documents'][0]
metadatas = search_results['metadatas'][0]

In [ ]:
# We inject metadata directly into the context so the LLM can "see" the sources
formatted_chunks = []
for i, (chunk, meta) in enumerate(zip(chunks, metadatas)):
    header = f"--- SOURCE {i+1} [File: {meta.get('source')}, Page: {meta.get('page')}] ---"
    formatted_chunks.append(f"{header}\n{chunk}")


In [ ]:
try:
    answer = engine.generate_answer(raw_question, formatted_chunks)

    print(f"\n" + "="*50)
    print(f"SAFETY ENGINEER RESPONSE")
    print("="*50 + "\n")
    print(answer)
finally:
    # --- STAFF LEVEL MEMORY MANAGEMENT ---
    # This forces the GPU to release temporary "activation" memory
    torch.cuda.empty_cache()
    print("\n[System Info] GPU VRAM cache cleared.")

In [ ]:
print("\n" + "-"*30)
print("SOURCES CONSULTED:")
for meta in metadatas:
    print(f"- {meta.get('source')} (Page {meta.get('page')})")
print("-"*30)